# 🚀 FastAPI — Serve MLflow Model & Manage Config via REST API

1. Load the registered model from MLflow
2. Create a FastAPI app with a `/predict` POST endpoint
3. Run the server in background
4. Test it with a sample POST request
5. Run the app via `uvicorn` CLI (standalone `.py` file)
6. FastAPI HTTP methods explained — Managing Metric Thresholds in `config.json`

---

In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
import requests
import threading
import time
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

/opt/anaconda3/envs/mlenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1️⃣ Load Model from MLflow Registry

In [2]:
MODEL_NAME    = "rf_pipeline_model"
MODEL_VERSION = "1"

model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{MODEL_VERSION}")

print(f"Loaded: {MODEL_NAME} v{MODEL_VERSION}")
print(f"Type  : {type(model).__name__}")
print(f"Steps : {list(model.named_steps.keys())}")

Loaded: rf_pipeline_model v1
Type  : Pipeline
Steps : ['preprocessor', 'classifier']


---
## 2️⃣ Create FastAPI App with `/predict` POST Endpoint

In [3]:
# --- Request / Response schemas ---
class PredictRequest(BaseModel):
    Quantity: float
    UnitPrice: float
    Country: str

class PredictResponse(BaseModel):
    prediction: int
    label: str

# --- FastAPI app ---
app = FastAPI(title="MLflow Model API")

@app.get("/")
def root():
    return {"status": "running", "model": MODEL_NAME, "version": MODEL_VERSION}

@app.post("/predict", response_model=PredictResponse)
def predict(data: PredictRequest):
    df = pd.DataFrame([data.model_dump()])
    pred = int(model.predict(df)[0])
    label = "High-Value" if pred == 1 else "Low-Value"
    return PredictResponse(prediction=pred, label=label)

print("✅ FastAPI app created with /predict endpoint")

✅ FastAPI app created with /predict endpoint


---
## 3️⃣ Run the Server (background thread)

In [4]:
# Run uvicorn in a background thread so the notebook stays interactive
thread = threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "127.0.0.1", "port": 8000, "log_level": "warning"},
    daemon=True
)
thread.start()
time.sleep(2)  # give the server a moment to start

print("🚀 Server running at http://127.0.0.1:8000")
print("📖 Docs at        http://127.0.0.1:8000/docs")

🚀 Server running at http://127.0.0.1:8000
📖 Docs at        http://127.0.0.1:8000/docs


---
## 4️⃣ Test with a Sample POST Request

In [5]:
sample = {
    "Quantity": 6,
    "UnitPrice": 2.55,
    "Country": "United Kingdom"
}

response = requests.post("http://127.0.0.1:8000/predict", json=sample)

print(f"Status Code : {response.status_code}")
print(f"Response    : {response.json()}")

Status Code : 200
Response    : {'prediction': 1, 'label': 'High-Value'}


---
## 5️⃣ Alternative — Run via `uvicorn` CLI

Write the app to a standalone `.py` file and run it from the terminal using `uvicorn`.

In [9]:
%%writefile app.py
# pyrefly: ignore [missing-import]
import json
from pathlib import Path
from typing import Dict
import mlflow.sklearn
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

# ── Load model at startup ──
MODEL_NAME = "rf_pipeline_model"
MODEL_VERSION = "1"
model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{MODEL_VERSION}")

# ── Config File Setup ──
CONFIG_PATH = Path("config.json")
DEFAULT_THRESHOLDS = {
    "accuracy": 0.85,
    "precision": 0.80,
    "recall": 0.75,
    "f1_score": 0.78,
    "latency_ms": 200.0,
}

def load_config() -> dict:
    if not CONFIG_PATH.exists():
        save_config(DEFAULT_THRESHOLDS)
    with open(CONFIG_PATH, "r") as f:
        return json.load(f)

def save_config(data: dict):
    with open(CONFIG_PATH, "w") as f:
        json.dump(data, f, indent=4)

# ── Schemas ──
class PredictRequest(BaseModel):
    Quantity: float
    UnitPrice: float
    Country: str

class PredictResponse(BaseModel):
    prediction: int
    label: str

class AddMetricThreshold(BaseModel):
    metric: str = Field(..., description="Metric name, e.g., 'roc_auc'")
    threshold: float = Field(..., description="Threshold value, e.g., 0.88")

# ── App ──
app = FastAPI(title="MLflow Model & Metric Thresholds API")

# Root endpoint
@app.get("/")
def root():
    return {
        "status": "running",
        "model": MODEL_NAME,
        "version": MODEL_VERSION,
        "endpoints": {
            "GET /config": "Read all metric thresholds from config file",
            "GET /config/{metric}": "Read threshold for a specific metric",
            "POST /config": "Add a new metric threshold to config file",
            "PUT /config": "Full update / replace all metric thresholds in config file",
            "PATCH /config": "Partial update of specific metric thresholds in config file",
            "DELETE /config/{metric}": "Remove a metric threshold from config file",
            "POST /predict": "Run model prediction"
        }
    }

# POST — ML Model Prediction
@app.post("/predict", response_model=PredictResponse)
def predict(data: PredictRequest):
    df = pd.DataFrame([data.model_dump()])
    pred = int(model.predict(df)[0])
    label = "High-Value" if pred == 1 else "Low-Value"
    return PredictResponse(prediction=pred, label=label)

# ── HTTP METHODS ON CONFIG FILE (METRIC THRESHOLDS) ──

# 1. GET — Read all thresholds or a specific metric threshold from config.json
@app.get("/config")
def get_all_thresholds():
    """Read and return all metric thresholds from config.json"""
    config = load_config()
    return {"status": "success", "thresholds": config}

@app.get("/config/{metric}")
def get_metric_threshold(metric: str):
    """Read threshold for a specific metric from config.json"""
    config = load_config()
    if metric not in config:
        raise HTTPException(status_code=404, detail=f"Metric '{metric}' not found in config.json")
    return {"metric": metric, "threshold": config[metric]}

# 2. POST — Create / Add a new metric threshold into config.json
@app.post("/config")
def add_metric_threshold(data: AddMetricThreshold):
    """Add a new metric threshold to config.json (fails if metric already exists)"""
    config = load_config()
    if data.metric in config:
        raise HTTPException(status_code=400, detail=f"Metric '{data.metric}' already exists in config.json. Use PUT or PATCH to update.")
    config[data.metric] = data.threshold
    save_config(config)
    return {"message": f"Added metric threshold for '{data.metric}'", "config": config}

# 3. PUT — Full Update / Replace entire config.json with a new set of thresholds
@app.put("/config")
def replace_all_thresholds(data: Dict[str, float]):
    """Replace all metric thresholds in config.json with the provided dictionary"""
    if not data:
        raise HTTPException(status_code=400, detail="Cannot replace config with an empty dictionary.")
    save_config(data)
    return {"message": "Entire thresholds config replaced successfully", "config": data}

# 4. PATCH — Partial Update: modify only specified metric thresholds without altering others
@app.patch("/config")
def patch_thresholds(data: Dict[str, float]):
    """Partially update only the provided metric thresholds in config.json"""
    config = load_config()
    for key, value in data.items():
        config[key] = value
    save_config(config)
    return {"message": "Updated specified metric thresholds", "config": config}

# 5. DELETE — Remove a metric threshold from config.json
@app.delete("/config/{metric}")
def delete_metric_threshold(metric: str):
    """Delete a specific metric threshold from config.json"""
    config = load_config()
    if metric not in config:
        raise HTTPException(status_code=404, detail=f"Metric '{metric}' not found in config.json")
    removed_val = config.pop(metric)
    save_config(config)
    return {"message": f"Deleted metric '{metric}' with threshold {removed_val}", "config": config}


Overwriting app.py


Now open a terminal and run:

```bash
uvicorn app:app --host 127.0.0.1 --port 8080 --reload
```

Then test from another terminal or from this notebook 👇

In [10]:
# Test the uvicorn CLI server (run this after starting the server in terminal)
sample = {
    "Quantity": 6,
    "UnitPrice": 2.55,
    "Country": "United Kingdom"
}

response = requests.post("http://127.0.0.1:8080/predict", json=sample)

print(f"Status Code : {response.status_code}")
print(f"Response    : {response.json()}")

Status Code : 200
Response    : {'prediction': 1, 'label': 'High-Value'}


---
## 6️⃣ FastAPI HTTP Methods — Managing Metric Thresholds in `config.json`

In production ML systems, we maintain **performance and operational thresholds** for metrics (e.g. minimum accuracy, precision, maximum latency SLA) inside a configuration file (`config.json`). This allows dynamic tuning without redeploying code.

| Method   | Purpose                        | Endpoint                   | Body Required? |
|----------|--------------------------------|----------------------------|----------------|
| **GET**    | Read / retrieve thresholds     | `/config` or `/config/{metric}` | No             |
| **POST**   | Create / add a new threshold   | `/config`                  | Yes (`{"metric": "...", "threshold": ...}`) |
| **PUT**    | Full update (replace entirely) | `/config`                  | Yes (complete config dictionary) |
| **PATCH**  | Partial update (some fields)   | `/config`                  | Yes (only fields to modify) |
| **DELETE** | Remove a metric threshold      | `/config/{metric}`         | No             |

**Make sure the server is running on port 8080 before running the cells below.**

### 🔄 REST Mapping: CRUD Operations on `config.json`

| CRUD Operation | HTTP Method | Action on `config.json` | Real-World MLOps Use Case |
|:---|:---:|:---|:---|
| **CREATE** | **POST** | Add a new metric threshold | Introduce monitoring for a new metric (e.g., `roc_auc: 0.88`) |
| **READ**   | **GET**  | Retrieve current thresholds | Continuous monitoring agent fetches baseline validation gates |
| **UPDATE (Full)** | **PUT** | Replace all thresholds entirely | Promote model to production: enforce strict SLA policy across all metrics |
| **UPDATE (Partial)** | **PATCH** | Update specific metric thresholds | Relax or tighten a single metric (e.g., `latency_ms: 120.0`) |
| **DELETE** | **DELETE** | Remove a metric threshold | Retire a deprecated metric no longer evaluated |

### 🟢 GET — Read Metric Thresholds from `config.json` (No Body Required)

Used to **retrieve** current threshold configurations without modifying server state.
- **Safe & Idempotent**: Calling GET multiple times produces no side effects.
- `GET /config`: Fetches all metric thresholds stored in `config.json`.
- `GET /config/{metric}`: Queries the threshold for a specific metric (e.g. `accuracy`).

In [11]:
BASE = "http://127.0.0.1:8080"

# 1. GET /config — Retrieve all metric thresholds
resp = requests.get(f"{BASE}/config")
print("GET /config:")
print(f"  Status Code : {resp.status_code}")
print(f"  Thresholds  : {resp.json()['thresholds']}")

# 2. GET /config/{metric} — Retrieve a single metric threshold
resp_acc = requests.get(f"{BASE}/config/accuracy")
print("\nGET /config/accuracy:")
print(f"  Status Code : {resp_acc.status_code}")
print(f"  Result      : {resp_acc.json()}")


GET /config:
  Status Code : 200
  Thresholds  : {'accuracy': 0.85, 'precision': 0.8, 'recall': 0.75, 'f1_score': 0.78, 'latency_ms': 200.0}

GET /config/accuracy:
  Status Code : 200
  Result      : {'metric': 'accuracy', 'threshold': 0.85}


### 🟡 POST — Create a New Metric Threshold in `config.json`

Used to **create / add** a new resource (here: a new metric threshold `roc_auc = 0.88`).
- **Requires Request Body**: `{"metric": "roc_auc", "threshold": 0.88}`
- **Non-idempotent**: Calling POST again with the same metric name returns `400 Bad Request` because the key already exists (use PUT or PATCH to update).

In [12]:
# POST /config — Add a new metric threshold to config.json
new_metric = {
    "metric": "roc_auc",
    "threshold": 0.88
}

resp = requests.post(f"{BASE}/config", json=new_metric)
print("POST /config:")
print(f"  Status Code : {resp.status_code}")
print(f"  Response    : {resp.json()}")

# Verify it was persisted to config.json
verify = requests.get(f"{BASE}/config/roc_auc")
print(f"\nVerification GET /config/roc_auc -> {verify.json()}")


POST /config:
  Status Code : 200
  Response    : {'message': "Added metric threshold for 'roc_auc'", 'config': {'accuracy': 0.85, 'precision': 0.8, 'recall': 0.75, 'f1_score': 0.78, 'latency_ms': 200.0, 'roc_auc': 0.88}}

Verification GET /config/roc_auc -> {'metric': 'roc_auc', 'threshold': 0.88}


### 🔵 PUT — Full Update (Replace Entire `config.json`)

Used to **completely replace** the existing configuration file with an updated set of thresholds.
- **Requires Full Body**: Must provide the complete dictionary. Any previously existing metric omitted from this payload is removed.
- **Idempotent**: Sending the same PUT payload multiple times yields the exact same state.
- **Use Case**: Applying an upgraded production policy across all metrics simultaneously.

In [13]:
# PUT /config — Full replacement of config.json with stricter production thresholds
production_policy = {
    "accuracy": 0.90,
    "precision": 0.85,
    "recall": 0.80,
    "f1_score": 0.82,
    "latency_ms": 150.0,
    "roc_auc": 0.91
}

resp = requests.put(f"{BASE}/config", json=production_policy)
print("PUT /config:")
print(f"  Status Code : {resp.status_code}")
print(f"  Message     : {resp.json()['message']}")
print(f"  New Config  : {resp.json()['config']}")


PUT /config:
  Status Code : 200
  Message     : Entire thresholds config replaced successfully
  New Config  : {'accuracy': 0.9, 'precision': 0.85, 'recall': 0.8, 'f1_score': 0.82, 'latency_ms': 150.0, 'roc_auc': 0.91}


### 🟠 PATCH — Partial Update (Modify Specific Metric Thresholds)

Used to **partially update** specific metric thresholds without touching the remaining config.
- **Difference from PUT**: `PUT` overwrites all keys; `PATCH` modifies only the supplied keys and preserves everything else.
- Here we adjust only `latency_ms` and `recall` — `accuracy`, `precision`, `f1_score`, and `roc_auc` remain untouched.

In [14]:
# PATCH /config — Update only latency_ms and recall without modifying other metrics
partial_update = {
    "latency_ms": 120.0,
    "recall": 0.83
}

resp = requests.patch(f"{BASE}/config", json=partial_update)
print("PATCH /config:")
print(f"  Status Code : {resp.status_code}")
print(f"  Message     : {resp.json()['message']}")
print(f"  Config      : {resp.json()['config']}")


PATCH /config:
  Status Code : 200
  Message     : Updated specified metric thresholds
  Config      : {'accuracy': 0.9, 'precision': 0.85, 'recall': 0.83, 'f1_score': 0.82, 'latency_ms': 120.0, 'roc_auc': 0.91}


### 🔴 DELETE — Remove a Metric Threshold from `config.json`

Used to **remove** a metric threshold from the configuration file (e.g. retiring `roc_auc`).
- **Endpoint**: `DELETE /config/{metric}`
- **Verification**: Querying the deleted metric afterwards yields `404 Not Found`.

In [15]:
# DELETE /config/roc_auc — Delete the roc_auc metric threshold from config.json
resp = requests.delete(f"{BASE}/config/roc_auc")
print("DELETE /config/roc_auc:")
print(f"  Status Code : {resp.status_code}")
print(f"  Message     : {resp.json()['message']}")

# Verification: Attempting to retrieve the deleted metric returns 404 Not Found
verify = requests.get(f"{BASE}/config/roc_auc")
print(f"\nVerification GET /config/roc_auc:")
print(f"  Status Code : {verify.status_code} (Expected 404)")
print(f"  Detail      : {verify.json()['detail']}")

# Inspect final config.json
final = requests.get(f"{BASE}/config").json()
print(f"\nFinal config.json: {final['thresholds']}")


DELETE /config/roc_auc:
  Status Code : 200
  Message     : Deleted metric 'roc_auc' with threshold 0.91

Verification GET /config/roc_auc:
  Status Code : 404 (Expected 404)
  Detail      : Metric 'roc_auc' not found in config.json

Final config.json: {'accuracy': 0.9, 'precision': 0.85, 'recall': 0.83, 'f1_score': 0.82, 'latency_ms': 120.0}
